# Model wczesnego ostrzegania — frekwencja pojedynczego ucznia
Wczytuje `zbior_ml_frekwencja_uczen.parquet` (wyeksportowany z `analiza_v2.ipynb`) i trenuje modele przewidujące frekwencję **konkretnego ucznia** w nadchodzącym miesiącu. Bez EDA, bez dashboardu.


## Import bibliotek

In [14]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone

## Wczytanie danych
Zbiór: 1 wiersz = (rok szkolny, uczeń, miesiąc). `uczen_id` służy tylko do identyfikacji — NIE jest cechą modelu.


In [2]:
SCIEZKA_DANYCH = "zbior_ml_frekwencja_uczen.parquet"

zbior_ml = pd.read_parquet(SCIEZKA_DANYCH)
print(f"Wczytano: {zbior_ml.shape[0]} wierszy, {zbior_ml.shape[1]} kolumn, {zbior_ml['uczen_id'].nunique()} uczniów")
zbior_ml.head(10)

Wczytano: 21904 wierszy, 22 kolumn, 856 uczniów


,rok_szkolny,uczen_id,Dziennik,rocznik,sufiks,numer_klasy,miesiac_szkolny_nr,miesiac_nazwa,pora_roku,wszystkich,...,liczba_miesiecy_danych_wczesniej,frekwencja_poprzedni_miesiac,frekwencja_2_miesiace_temu,frekwencja_srednia_od_poczatku_roku,frekwencja_odchylenie_dotychczas,frekwencja_min_dotychczas,frekwencja_trend_2m,liczba_miesiecy_ponizej_progu,frekwencja_klasy_poprzedni_miesiac,frekwencja_pct
0,2020/2021,0023EFDB2E,2RŻ5,2019,RŻ5,2,1,wrzesień,jesień,95,...,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,100.0
1,2020/2021,0023EFDB2E,2RŻ5,2019,RŻ5,2,2,październik,jesień,85,...,1,100.0,NaN,100.000000,NaN,100.0,NaN,0.0,97.065667,100.0
2,2020/2021,0023EFDB2E,2RŻ5,2019,RŻ5,2,3,listopad,jesień,0,...,2,100.0,100.0,100.000000,0.000000,100.0,0.0,0.0,92.181667,NaN
3,2020/2021,0023EFDB2E,2RŻ5,2019,RŻ5,2,4,grudzień,zima,0,...,3,NaN,100.0,100.000000,0.000000,100.0,NaN,0.0,0.000000,NaN
4,2020/2021,0023EFDB2E,2RŻ5,2019,RŻ5,2,5,styczeń,zima,0,...,4,NaN,NaN,100.000000,0.000000,100.0,NaN,0.0,0.000000,NaN
5,2020/2021,0023EFDB2E,2RŻ5,2019,RŻ5,2,6,luty,zima,2,...,5,NaN,NaN,100.000000,0.000000,100.0,NaN,0.0,0.000000,50.0
6,2020/2021,0023EFDB2E,2RŻ5,2019,RŻ5,2,7,marzec,wiosna,0,...,6,50.0,NaN,83.333333,28.867513,50.0,NaN,1.0,36.954074,NaN
7,2020/2021,0023EFDB2E,2RŻ5,2019,RŻ5,2,8,kwiecień,wiosna,0,...,7,NaN,50.0,83.333333,28.867513,50.0,NaN,1.0,0.000000,NaN
8,2020/2021,0023EFDB2E,2RŻ5,2019,RŻ5,2,9,maj,wiosna,29,...,8,NaN,NaN,83.333333,28.867513,50.0,NaN,1.0,0.000000,100.0
9,2020/2021,0023EFDB2E,2RŻ5,2019,RŻ5,2,10,czerwiec,wiosna,22,...,9,100.0,NaN,87.500000,25.000000,50.0,NaN,1.0,89.646000,100.0


## Przygotowanie danych
Odrzucamy wiersze bez pełnej historii ucznia (początek roku szkolnego). `frekwencja_trend_2m` wypełniamy zerem tam, gdzie brakuje drugiego miesiąca histori.


In [3]:
# Odrzuć wiersze bez pełnej historii ORAZ wiersze z NaN w targecie (np. uczeń z 0 lekcji w danym
# miesiącu - dzielenie 0/0 przy liczeniu frekwencja_pct, zdarza się np. przy dłuższej chorobie/transferze)
przed = len(zbior_ml)
liczba_zero_lekcji = (zbior_ml["wszystkich"] == 0).sum()
if liczba_zero_lekcji:
    print(f"Uwaga: {liczba_zero_lekcji} wierszy ma 0 lekcji w danym miesiącu (frekwencja_pct = NaN) - odrzucam.")

# Nowe cechy wymagają więcej historii (np. 2_miesiace_temu potrzebuje aż 2 poprzednich miesięcy) -
# odrzucamy więc więcej wierszy na start roku niż poprzednio, ale to konieczne dla tych cech
zbior_model = zbior_ml.dropna(
    subset=["frekwencja_poprzedni_miesiac", "frekwencja_2_miesiace_temu",
            "frekwencja_srednia_od_poczatku_roku", "frekwencja_odchylenie_dotychczas",
            "frekwencja_pct"]
).copy()
zbior_model["frekwencja_trend_2m"] = zbior_model["frekwencja_trend_2m"].fillna(0)
zbior_model["zmiana_klasy"] = zbior_model["zmiana_klasy"].astype(int)
# frekwencja_klasy_poprzedni_miesiac może brakować, jeśli cała klasa nie miała danych poprzedniego miesiąca -
# rzadkie, ale wypełniamy średnią z reszty zbioru zamiast tracić wiersz
zbior_model["frekwencja_klasy_poprzedni_miesiac"] = zbior_model["frekwencja_klasy_poprzedni_miesiac"].fillna(
    zbior_model["frekwencja_klasy_poprzedni_miesiac"].mean()
)

print(f"Wierszy po odrzuceniu braków historii i braków w targecie: {len(zbior_model)} (z {przed})")
assert zbior_model[["frekwencja_poprzedni_miesiac", "frekwencja_srednia_od_poczatku_roku",
                     "frekwencja_trend_2m", "frekwencja_pct"]].isna().sum().sum() == 0, "nadal są NaN - sprawdź dane"

cechy_liczbowe = [
    "numer_klasy", "miesiac_szkolny_nr", "wszystkich", "liczba_uczniow_w_klasie", "zmiana_klasy",
    "liczba_miesiecy_danych_wczesniej",
    "frekwencja_poprzedni_miesiac", "frekwencja_2_miesiace_temu",
    "frekwencja_srednia_od_poczatku_roku", "frekwencja_odchylenie_dotychczas", "frekwencja_min_dotychczas",
    "frekwencja_trend_2m", "liczba_miesiecy_ponizej_progu",
    "frekwencja_klasy_poprzedni_miesiac",
]
cechy_kategoryczne = ["sufiks", "pora_roku"]
target = "frekwencja_pct"

# uczen_id CELOWO nie wchodzi do cech - służy tylko do grupowania przy podziale train/test poniżej

Uwaga: 623 wierszy ma 0 lekcji w danym miesiącu (frekwencja_pct = NaN) - odrzucam.
Wierszy po odrzuceniu braków historii i braków w targecie: 16190 (z 21904)


## Podział train/test — PO CZASIE, nie losowo
Ostatni dostępny rok szkolny = test. Losowy podział wyciekłby informację między sąsiednimi miesiącami TEGO SAMEGO ucznia (jeszcze silniejszy efekt niż przy danych klasowych).


In [4]:
lata_dostepne = sorted(zbior_model["rok_szkolny"].unique())
rok_testowy = lata_dostepne[-1]

train = zbior_model[zbior_model["rok_szkolny"] != rok_testowy]
test = zbior_model[zbior_model["rok_szkolny"] == rok_testowy]

print(f"Trening: lata {lata_dostepne[:-1]} ({len(train)} wierszy, {train['uczen_id'].nunique()} uczniów)")
print(f"Test: rok {rok_testowy} ({len(test)} wierszy, {test['uczen_id'].nunique()} uczniów)")

X_train, y_train = train[cechy_liczbowe + cechy_kategoryczne], train[target]
X_test, y_test = test[cechy_liczbowe + cechy_kategoryczne], test[target]

Trening: lata ['2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025'] (12933 wierszy, 708 uczniów)
Test: rok 2025/2026 (3257 wierszy, 434 uczniów)


## Baseline: "będzie tak jak w poprzednim miesiącu"


In [5]:
baseline_pred = X_test["frekwencja_poprzedni_miesiac"]
mae_baseline = mean_absolute_error(y_test, baseline_pred)
rmse_baseline = mean_squared_error(y_test, baseline_pred) ** 0.5
r2_baseline = r2_score(y_test, baseline_pred)
print(f"Baseline — MAE: {mae_baseline:.2f} | RMSE: {rmse_baseline:.2f} | R²: {r2_baseline:.3f}")

Baseline — MAE: 10.05 | RMSE: 18.82 | R²: 0.220


## Model 1: Ridge (regresja liniowa z regularyzacją)

In [6]:
preprocessing = ColumnTransformer([
    ("liczby", StandardScaler(), cechy_liczbowe),
    ("kategorie", OneHotEncoder(handle_unknown="ignore"), cechy_kategoryczne),
])

model_ridge = Pipeline([("prep", preprocessing), ("model", Ridge(alpha=1.0))])
model_ridge.fit(X_train, y_train)

pred_ridge = model_ridge.predict(X_test)
pred_ridge = np.clip(pred_ridge, 0, 100)  # frekwencja to procent - regresja liniowa może "wystrzelić" poza [0,100]
mae_ridge = mean_absolute_error(y_test, pred_ridge)
rmse_ridge = mean_squared_error(y_test, pred_ridge) ** 0.5
r2_ridge = r2_score(y_test, pred_ridge)

print(f"Ridge — MAE: {mae_ridge:.2f} | RMSE: {rmse_ridge:.2f} | R²: {r2_ridge:.3f}")

Ridge — MAE: 7.23 | RMSE: 11.53 | R²: 0.707


## Model 2: Random Forest

In [7]:
model_rf = Pipeline([("prep", preprocessing), ("model", RandomForestRegressor(
    n_estimators=300, max_depth=6, min_samples_leaf=10, random_state=42
))])
model_rf.fit(X_train, y_train)

pred_rf = model_rf.predict(X_test)
pred_rf = np.clip(pred_rf, 0, 100)  # Random Forest zwykle i tak mieści się w zakresie, ale dla spójności przycinamy też tutaj
mae_rf = mean_absolute_error(y_test, pred_rf)
rmse_rf = mean_squared_error(y_test, pred_rf) ** 0.5
r2_rf = r2_score(y_test, pred_rf)

print(f"Random Forest — MAE: {mae_rf:.2f} | RMSE: {rmse_rf:.2f} | R²: {r2_rf:.3f}")

Random Forest — MAE: 6.80 | RMSE: 11.35 | R²: 0.716


## Porównanie wszystkich modeli z baseline

In [8]:
wyniki = pd.DataFrame({
    "Model": ["Baseline (poprzedni miesiąc)", "Ridge", "Random Forest"],
    "MAE [pkt proc.]": [round(mae_baseline, 2), round(mae_ridge, 2), round(mae_rf, 2)],
    "RMSE [pkt proc.]": [round(rmse_baseline, 2), round(rmse_ridge, 2), round(rmse_rf, 2)],
    "R²": [round(r2_baseline, 3), round(r2_ridge, 3), round(r2_rf, 3)],
})
wyniki["Lepszy od baseline? (MAE)"] = ["—", *(f"{'TAK' if m < mae_baseline else 'nie'} ({mae_baseline - m:+.2f})"
                                              for m in [mae_ridge, mae_rf])]
wyniki

,Model,MAE [pkt proc.],RMSE [pkt proc.],R²,Lepszy od baseline? (MAE)
0,Baseline (poprzedni miesiąc),10.05,18.82,0.220,—
1,Ridge,7.23,11.53,0.707,TAK (+2.82)
2,Random Forest,6.80,11.35,0.716,TAK (+3.25)


## Czy wynik jest stabilny? Walidacja krzyżowa w czasie
Pojedynczy podział train/test (ostatni rok = test) mógł trafić akurat na łatwiejszy albo trudniejszy rok. Tutaj powtarzamy cały proces dla **każdego możliwego roku jako testowego** (zaczynając od roku, przed którym jest przynajmniej 2 lata historii), zawsze trenując na wszystkich latach wcześniejszych ('expanding window' - dokładnie tak, jak model realnie działałby w kolejnych latach). Jeśli MAE/R² są podobne we wszystkich foldach - wynik jest wiarygodny. Jeśli mocno skaczą - pojedynczy podział z poprzednich komórek nie był reprezentatywny.


In [ ]:


MIN_LAT_TRENINGOWYCH = 2  # potrzebujemy min. tylu lat historii, zanim zaczniemy testować

lata_wszystkie = sorted(zbior_model["rok_szkolny"].unique())
wyniki_cv = []

for i in range(MIN_LAT_TRENINGOWYCH, len(lata_wszystkie)):
    rok_test = lata_wszystkie[i]
    lata_train = lata_wszystkie[:i]

    train_cv = zbior_model[zbior_model["rok_szkolny"].isin(lata_train)]
    test_cv = zbior_model[zbior_model["rok_szkolny"] == rok_test]
    if len(train_cv) == 0 or len(test_cv) == 0:
        continue

    X_train_cv = train_cv[cechy_liczbowe + cechy_kategoryczne]
    y_train_cv = train_cv[target]
    X_test_cv = test_cv[cechy_liczbowe + cechy_kategoryczne]
    y_test_cv = test_cv[target]

    model_ridge_cv = clone(model_ridge)
    model_ridge_cv.fit(X_train_cv, y_train_cv)
    pred_ridge_cv = np.clip(model_ridge_cv.predict(X_test_cv), 0, 100)

    model_rf_cv = clone(model_rf)
    model_rf_cv.fit(X_train_cv, y_train_cv)
    pred_rf_cv = np.clip(model_rf_cv.predict(X_test_cv), 0, 100)

    baseline_pred_cv = X_test_cv["frekwencja_poprzedni_miesiac"]

    wyniki_cv.append({
        "rok_testowy": rok_test,
        "lat_treningowych": len(lata_train),
        "n_train": len(train_cv), "n_test": len(test_cv),
        "mae_baseline": mean_absolute_error(y_test_cv, baseline_pred_cv),
        "mae_ridge": mean_absolute_error(y_test_cv, pred_ridge_cv),
        "mae_rf": mean_absolute_error(y_test_cv, pred_rf_cv),
        "r2_ridge": r2_score(y_test_cv, pred_ridge_cv),
        "r2_rf": r2_score(y_test_cv, pred_rf_cv),
    })

df_cv = pd.DataFrame(wyniki_cv).round(3)
print(df_cv.to_string(index=False))
print()
print("Podsumowanie na przestrzeni wszystkich foldów (średnia ± odchylenie standardowe):")
for kol, etykieta in [("mae_baseline", "MAE baseline"), ("mae_ridge", "MAE Ridge"), ("mae_rf", "MAE Random Forest"),
                       ("r2_ridge", "R² Ridge"), ("r2_rf", "R² Random Forest")]:
    print(f"  {etykieta}: {df_cv[kol].mean():.2f} ± {df_cv[kol].std():.2f}")

if df_cv["mae_rf"].std() > df_cv["mae_rf"].mean() * 0.3:
    print("\n⚠️  Duży rozrzut MAE między latami (std > 30% średniej) - pojedynczy podział train/test")
    print("   z poprzednich komórek mógł nie być reprezentatywny dla typowej skuteczności modelu.")
else:
    print("\n✅ MAE względnie stabilne między latami - wynik z pojedynczego podziału wcześniej wygląda wiarygodnie.")

rok_testowy  lat_treningowych  n_train  n_test  mae_baseline  mae_ridge  mae_rf  r2_ridge  r2_rf
  2022/2023                 2     3855    2804         7.541      6.827   6.166     0.536  0.593
  2023/2024                 3     6659    3136         8.613      7.051   6.159     0.683  0.729
  2024/2025                 4     9795    3138         8.254      7.064   6.499     0.706  0.733
  2025/2026                 5    12933    3257        10.051      7.233   6.805     0.707  0.716

Podsumowanie na przestrzeni wszystkich foldów (średnia ± odchylenie standardowe):
  MAE baseline: 8.61 ± 1.06
  MAE Ridge: 7.04 ± 0.17
  MAE Random Forest: 6.41 ± 0.31
  R² Ridge: 0.66 ± 0.08
  R² Random Forest: 0.69 ± 0.07

✅ MAE względnie stabilne między latami - wynik z pojedynczego podziału wcześniej wygląda wiarygodnie.


## Jakość alertów: macierz pomyłek
MAE/RMSE/R² mówią o błędzie prognozy jako liczby, ale nie odpowiadają wprost na pytanie "czy model łapie zagrożonych uczniów". Tutaj binaryzujemy predykcję i rzeczywistość na zbiorze **testowym** (rok, którego model nie widział podczas treningu) wg tego samego progu co w liście alertów poniżej, i sprawdzamy: ilu faktycznie zagrożonych uczniów model wyłapał (recall), a ile alertów to fałszywe alarmy (precision).


In [10]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

PROG_ZAGROZENIA = 75.0  # ten sam próg co w liście alertów niżej - zmień w obu miejscach jeśli zmieniasz

rzeczywiscie_zagrozony = (y_test < PROG_ZAGROZENIA).astype(int)

def raport_jakosci_alertow(nazwa_modelu, predykcje):
    przewidywany_zagrozony = (predykcje < PROG_ZAGROZENIA).astype(int)
    cm = confusion_matrix(rzeczywiscie_zagrozony, przewidywany_zagrozony)
    tn, fp, fn, tp = cm.ravel()
    precision = precision_score(rzeczywiscie_zagrozony, przewidywany_zagrozony, zero_division=0)
    recall = recall_score(rzeczywiscie_zagrozony, przewidywany_zagrozony, zero_division=0)
    f1 = f1_score(rzeczywiscie_zagrozony, przewidywany_zagrozony, zero_division=0)

    print(f"=== {nazwa_modelu} (próg: {PROG_ZAGROZENIA}%) ===")
    print(f"Rzeczywiście zagrożonych w zbiorze testowym: {rzeczywiscie_zagrozony.sum()} / {len(rzeczywiscie_zagrozony)}")
    print()
    print(f"                    Przewidziano: OK   Przewidziano: zagrożony")
    print(f"Faktycznie: OK           {tn:>6}              {fp:>6}   <- fałszywe alarmy")
    print(f"Faktycznie: zagrożony    {fn:>6}   <- przeoczeni      {tp:>6}   <- trafione")
    print()
    print(f"Recall (ilu zagrożonych złapano):    {recall:.1%}  ({tp} z {tp+fn})")
    print(f"Precision (ile alertów trafnych):    {precision:.1%}  ({tp} z {tp+fp})")
    print(f"F1:                                  {f1:.2f}")
    print()
    return {"model": nazwa_modelu, "recall": recall, "precision": precision, "f1": f1, "tp": tp, "fp": fp, "fn": fn, "tn": tn}

wynik_ridge = raport_jakosci_alertow("Ridge", pred_ridge)
wynik_rf = raport_jakosci_alertow("Random Forest", pred_rf)

podsumowanie_alertow = pd.DataFrame([wynik_ridge, wynik_rf]).set_index("model")
podsumowanie_alertow[["recall", "precision", "f1"]] = podsumowanie_alertow[["recall", "precision", "f1"]].round(3)
podsumowanie_alertow

=== Ridge (próg: 75.0%) ===
Rzeczywiście zagrożonych w zbiorze testowym: 629 / 3257

                    Przewidziano: OK   Przewidziano: zagrożony
Faktycznie: OK             2491                 137   <- fałszywe alarmy
Faktycznie: zagrożony       162   <- przeoczeni         467   <- trafione

Recall (ilu zagrożonych złapano):    74.2%  (467 z 629)
Precision (ile alertów trafnych):    77.3%  (467 z 604)
F1:                                  0.76

=== Random Forest (próg: 75.0%) ===
Rzeczywiście zagrożonych w zbiorze testowym: 629 / 3257

                    Przewidziano: OK   Przewidziano: zagrożony
Faktycznie: OK             2504                 124   <- fałszywe alarmy
Faktycznie: zagrożony       163   <- przeoczeni         466   <- trafione

Recall (ilu zagrożonych złapano):    74.1%  (466 z 629)
Precision (ile alertów trafnych):    79.0%  (466 z 590)
F1:                                  0.76



,recall,precision,f1,tp,fp,fn,tn
model,,,,,,,
Ridge,0.742,0.773,0.758,467,137,162,2491
Random Forest,0.741,0.790,0.765,466,124,163,2504


## Interpretacja — co wpływa na prognozę

In [11]:
nazwy_cech = (
    cechy_liczbowe +
    list(model_ridge.named_steps["prep"].named_transformers_["kategorie"].get_feature_names_out(cechy_kategoryczne))
)

wspolczynniki = pd.Series(model_ridge.named_steps["model"].coef_, index=nazwy_cech).sort_values()
print("Współczynniki Ridge:")
print(wspolczynniki.to_string())

print()
waznosci = pd.Series(model_rf.named_steps["model"].feature_importances_, index=nazwy_cech).sort_values(ascending=False)
print("Ważność cech Random Forest:")
print(waznosci.to_string())

Współczynniki Ridge:
miesiac_szkolny_nr                     -3.917578
pora_roku_jesień                       -3.109566
sufiks_TW5                             -2.365887
sufiks_TR                              -2.278544
sufiks_RŻ5                             -2.157688
sufiks_TŻ5                             -2.099363
sufiks_TW                              -1.676643
sufiks_TM5                             -1.571555
liczba_miesiecy_ponizej_progu          -1.342410
sufiks_TR5                             -0.998501
sufiks_TŻ                              -0.935011
frekwencja_srednia_od_poczatku_roku    -0.801483
wszystkich                             -0.708274
zmiana_klasy                           -0.025876
sufiks_TMR                              0.026043
liczba_uczniow_w_klasie                 0.255389
pora_roku_zima                          0.468465
numer_klasy                             0.479281
frekwencja_trend_2m                     1.504946
sufiks_RŻ                               2.019422

## Lista uczniów zagrożonych w następnym miesiącu
Zamiast pojedynczej ręcznej predykcji — praktyczne zastosowanie: weź WSZYSTKICH uczniów z ostatniego dostępnego miesiąca w danych, przewidź im frekwencję na kolejny miesiąc, i pokaż posortowaną listę najbardziej zagrożonych — to dokładnie takie zestawienie wysłałbyś wychowawcom.


In [12]:
PROG_ZAGROZENIA = 75.0  # poniżej tej przewidywanej frekwencji - uczeń trafia na listę do sprawdzenia
MIN_LEKCJI_WIARYGODNE = 50  # miesiące z mniejszą liczbą lekcji ucznia pomijamy - % jest wtedy niewiarygodny
                             # (np. koniec 5-letniego cyklu, transfer w trakcie miesiąca - patrz wcześniejsza
                             # rozmowa o przypadku "Płoska Oliwia")

# WAŻNE: bierzemy tylko NAJNOWSZY rok szkolny, nie wszystkie lata naraz. Bez tego "ostatni miesiąc"
# per (rok_szkolny, uczeń) wyłapuje czerwiec KAŻDEGO roku dla KAŻDEGO ucznia - czyli miesza końcówki
# roku z 4 różnych lat w jedną listę, zamiast pokazać bieżący stan aktywnych uczniów. Zobacz rozmowę
# o tym, dlaczego miesiac_szkolny_nr=10 (czerwiec) dominował w poprzedniej wersji tej listy.
rok_biezacy = lata_dostepne[-1]
zbior_biezacy_rok = zbior_model[zbior_model["rok_szkolny"] == rok_biezacy]
print(f"Lista dotyczy tylko roku {rok_biezacy} - {zbior_biezacy_rok['uczen_id'].nunique()} uczniów")

ostatni_miesiac_kazdego_ucznia = (
    zbior_biezacy_rok.sort_values("miesiac_szkolny_nr")
    .groupby("uczen_id")  # <- grupujemy TYLKO po uczniu, nie po (rok_szkolny, uczeń) - w obrębie 1 roku to bez znaczenia
    .tail(1)
    .copy()
)
print(f"Rozkład miesiąca, z którego pochodzi 'ostatni znany' status (10=czerwiec, 8=kwiecień):")
print(ostatni_miesiac_kazdego_ucznia["miesiac_szkolny_nr"].value_counts().sort_index())

liczba_niewiarygodnych = (ostatni_miesiac_kazdego_ucznia["wszystkich"] < MIN_LEKCJI_WIARYGODNE).sum()
print(f"Uczniów, dla których ostatni znany miesiąc ma < {MIN_LEKCJI_WIARYGODNE} lekcji "
      f"(pomijamy jako niewiarygodne): {liczba_niewiarygodnych} z {len(ostatni_miesiac_kazdego_ucznia)}")

ostatni_miesiac_kazdego_ucznia = ostatni_miesiac_kazdego_ucznia[
    ostatni_miesiac_kazdego_ucznia["wszystkich"] >= MIN_LEKCJI_WIARYGODNE
].copy()

X_prognoza = ostatni_miesiac_kazdego_ucznia[cechy_liczbowe + cechy_kategoryczne]
ostatni_miesiac_kazdego_ucznia["prognoza_ridge"] = np.clip(model_ridge.predict(X_prognoza), 0, 100).round(1)
ostatni_miesiac_kazdego_ucznia["prognoza_rf"] = np.clip(model_rf.predict(X_prognoza), 0, 100).round(1)

# sprawdzenie: czy prognoza_rf faktycznie ma wartości (nie NaN) - gdyby coś nadal szwankowało, zobaczysz to od razu
assert ostatni_miesiac_kazdego_ucznia["prognoza_rf"].notna().all(), \
    "prognoza_rf ma braki - sprawdź czy model_rf był trenowany na tym samym zestawie cech co X_prognoza"

# --- Próg ABSOLUTNY (75%) - zostawiony do wglądu, ale w Twojej szkole obejmuje większość uczniów (63.8%!),
# więc jako "lista alertów" jest bezużyteczny - zbyt wielu uczniów, żeby to była praktyczna lista do sprawdzenia.
ponizej_progu_absolutnego = ostatni_miesiac_kazdego_ucznia["prognoza_rf"] < PROG_ZAGROZENIA
print(f"Próg absolutny {PROG_ZAGROZENIA}%: {ponizej_progu_absolutnego.sum()} uczniów "
      f"({ponizej_progu_absolutnego.mean():.1%} wszystkich) - zbyt dużo, żeby było to użyteczne jako alert")

# --- Próg WZGLĘDNY: dolne X% uczniów wg prognozy, niezależnie jaki jest ogólny poziom frekwencji w szkole ---
PROG_PERCENTYL = 0.10  # dolne 10% uczniów wg prognozowanej frekwencji - dostosuj do pojemności wychowawców

prog_dynamiczny = ostatni_miesiac_kazdego_ucznia["prognoza_rf"].quantile(PROG_PERCENTYL)
print(f"Próg względny (dolne {PROG_PERCENTYL:.0%}): odpowiada prognozowanej frekwencji {prog_dynamiczny:.1f}%")

zagrozeni = (
    ostatni_miesiac_kazdego_ucznia[ostatni_miesiac_kazdego_ucznia["prognoza_rf"] <= prog_dynamiczny]
    [["rok_szkolny", "uczen_id", "Dziennik", "wszystkich", "frekwencja_pct", "prognoza_ridge", "prognoza_rf"]]
    .rename(columns={"frekwencja_pct": "frekwencja_ostatni_znany_miesiac", "wszystkich": "lekcji_w_ostatnim_miesiacu"})
    .sort_values("prognoza_rf")
)
print(f"\nUczniowie w dolnych {PROG_PERCENTYL:.0%} wg prognozy: {len(zagrozeni)}")
zagrozeni

Lista dotyczy tylko roku 2025/2026 - 434 uczniów
Rozkład miesiąca, z którego pochodzi 'ostatni znany' status (10=czerwiec, 8=kwiecień):
miesiac_szkolny_nr
3       1
4       1
5       5
6       2
8      69
9       1
10    355
Name: count, dtype: int64
Uczniów, dla których ostatni znany miesiąc ma < 50 lekcji (pomijamy jako niewiarygodne): 112 z 434
Próg absolutny 75.0%: 274 uczniów (85.1% wszystkich) - zbyt dużo, żeby było to użyteczne jako alert
Próg względny (dolne 10%): odpowiada prognozowanej frekwencji 23.4%

Uczniowie w dolnych 10% wg prognozy: 34


,rok_szkolny,uczen_id,Dziennik,lekcji_w_ostatnim_miesiacu,frekwencja_ostatni_znany_miesiac,prognoza_ridge,prognoza_rf
20350,2025/2026,A2B9CCBA47,5TM5,75,5.33,0.0,9.5
20570,2025/2026,AD6BE5923B,3M,101,6.93,0.0,13.7
20957,2025/2026,C107B7F21D,3M,101,3.96,9.1,14.7
20081,2025/2026,907867E9B4,3M,103,9.71,0.0,14.9
19218,2025/2026,5C6CD666D0,5TM5,69,8.70,0.0,15.5
18096,2025/2026,10993B78D2,3M,103,22.33,18.6,15.6
20588,2025/2026,AE5812FBE6,5TM5,75,4.00,0.0,15.7
20302,2025/2026,9E8DAA8C13,5TM5,75,4.00,4.0,15.7
17909,2025/2026,0748FBFDF1,5TM5,75,8.00,0.4,15.9
18557,2025/2026,2EBEDF82B0,5TM5,73,9.59,5.7,16.2


## Wykres: rozkład prognozowanej frekwencji + dwa progi alertu
Zamiast jednego sztywnego progu (który, jak ustaliliśmy, obejmuje większość szkoły) - dwa poziomy: **krytyczny** (dolne 5%, do natychmiastowej interwencji) i **do obserwacji** (dolne 20%, szerszy nadzór). Histogram pokazuje gdzie te progi faktycznie wypadają na tle całego rozkładu.


In [13]:
import plotly.graph_objects as go

PROG_KRYTYCZNY = ostatni_miesiac_kazdego_ucznia["prognoza_rf"].quantile(0.05)
PROG_OBSERWACJA = ostatni_miesiac_kazdego_ucznia["prognoza_rf"].quantile(0.20)

liczba_krytyczni = (ostatni_miesiac_kazdego_ucznia["prognoza_rf"] <= PROG_KRYTYCZNY).sum()
liczba_obserwacja = (
    (ostatni_miesiac_kazdego_ucznia["prognoza_rf"] > PROG_KRYTYCZNY) &
    (ostatni_miesiac_kazdego_ucznia["prognoza_rf"] <= PROG_OBSERWACJA)
).sum()

print(f"Krytyczny (≤{PROG_KRYTYCZNY:.1f}%): {liczba_krytyczni} uczniów")
print(f"Do obserwacji ({PROG_KRYTYCZNY:.1f}–{PROG_OBSERWACJA:.1f}%): {liczba_obserwacja} uczniów")
print(f"Reszta (>{PROG_OBSERWACJA:.1f}%): {len(ostatni_miesiac_kazdego_ucznia) - liczba_krytyczni - liczba_obserwacja} uczniów")

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=ostatni_miesiac_kazdego_ucznia["prognoza_rf"],
    nbinsx=40, marker_color="#4C7A6B", opacity=0.85, name="Uczniowie"
))
fig.add_vline(x=PROG_KRYTYCZNY, line_dash="dash", line_color="#A8433A", line_width=2,
              annotation_text=f"Krytyczny: {PROG_KRYTYCZNY:.0f}%", annotation_position="top left")
fig.add_vline(x=PROG_OBSERWACJA, line_dash="dash", line_color="#C98A2C", line_width=2,
              annotation_text=f"Obserwacja: {PROG_OBSERWACJA:.0f}%", annotation_position="top left")
fig.add_vrect(x0=0, x1=PROG_KRYTYCZNY, fillcolor="#A8433A", opacity=0.08, line_width=0)
fig.add_vrect(x0=PROG_KRYTYCZNY, x1=PROG_OBSERWACJA, fillcolor="#C98A2C", opacity=0.08, line_width=0)

fig.update_layout(
    title="Rozkład prognozowanej frekwencji uczniów na kolejny miesiąc",
    xaxis_title="Prognozowana frekwencja [%]", yaxis_title="Liczba uczniów",
    height=450, showlegend=False,
    font=dict(family="Inter, sans-serif"),
    title_font=dict(family="Lora, serif", size=17),
)
fig.show()

Krytyczny (≤17.1%): 17 uczniów
Do obserwacji (17.1–35.5%): 48 uczniów
Reszta (>35.5%): 257 uczniów
